# Resultado CAMEL definitivo

Se supone que este notebook contendrá el proceso para extraer satisfactoriamente los resultados de la calificación CAMEL para las 88 cooperativas. Este proceso se desarrollará teniendo en cuenta el siguiente orden:

* Descarga y lectura de los datos.
* Extracción y valoración del PCA.
* Creación de los rangos (percentiles para las variables).
* Calificación de las variables.

# importacion de los datos

In [1]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import pandas as pd

In [2]:
datos=pd.read_excel("../tablas/registros_categorizados_con_IRL.xlsx")
datos.head()

,ID_registro,ID_indicador,ID_cooperativa,ano,mes,valor,categoria
0,25345,Quebranto Patrimonial,CAJA COOPERATIVA CREDICOOP,2020,1,0.0,Medianas
1,25346,Quebranto Patrimonial,CAJA COOPERATIVA PETROLERA,2020,1,0.0,Megas
2,25347,Quebranto Patrimonial,CASA NACIONAL DEL PROFESOR,2020,1,0.0,Megas
3,25348,Quebranto Patrimonial,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,2020,1,0.0,Grandes
4,25349,Quebranto Patrimonial,COOPERATIVA AHORRO Y CREDITO GOMEZ PLATA LTDA.,2020,1,0.0,Medianas


# Calculo PCA

In [3]:
def pesos_pca_grupo_coops(
    df,
    lista_coops,
    col_nombre="ID_cooperativa",
    n_componentes=3
):
    
    # filtrar cooperativas
    df_filtrado = df[df[col_nombre].isin(lista_coops)].copy()
    
    if df_filtrado.empty:
        raise ValueError("No hay datos para las cooperativas indicadas")
    
    # LIMPIEZA 
    df_filtrado["valor"] = (
        df_filtrado["valor"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    
    df_filtrado["valor"] = pd.to_numeric(df_filtrado["valor"], errors="coerce")
    
    # pivot (matriz indicadores)
    df_pivot = df_filtrado.pivot_table(
        index=col_nombre,
        columns="ID_indicador",
        values="valor",
        aggfunc="mean"
    )
    
    # imputar faltantes
    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(df_pivot)
    
    # escalar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)
    
    # PCA
    pca = PCA(n_components=min(n_componentes, df_pivot.shape[1]))
    pca.fit(X_scaled)
    
    loadings = pd.DataFrame(
        pca.components_.T,
        index=df_pivot.columns
    )
    
    pesos = (loadings**2).mean(axis=1)
    
    # porcentaje
    pesos = (pesos / pesos.sum()) * 100
    
    return pesos.sort_values(ascending=False)

# Calculo Rangos

In [5]:
# Tabla de los rangos
def tabla_percentiles_globales(df: pd.DataFrame):
    if df.empty:
        return pd.DataFrame()

    df = df.copy()

    # 🔹 Filtrar valores positivos (evita que los ceros dañen la distribución)
    df = df[df["valor"] > 0]

    if df.empty:
        return pd.DataFrame()

    # 🔹 Transformación logarítmica para datos sesgados
    df["valor_log"] = np.log1p(df["valor"])

    # 🔹 Percentiles reales
    percentiles = [i/10 for i in range(1, 11)]

    tabla = (
        df
        .groupby("ID_indicador")["valor_log"]
        .quantile(percentiles)
        .unstack()
    )

    # 🔹 Volver a escala original (opcional pero recomendado)
    tabla = np.expm1(tabla)

    # 🔹 Renombrar columnas
    tabla.columns = [f"P{int(p*100)}" for p in percentiles]

    return tabla

# Calcular calificacion de los indicadores 

In [6]:
def calcular_calificaciones_formato_lista(df: pd.DataFrame):
    if df.empty:
        return []

    columnas_necesarias = {
        'ID_indicador', 'ID_cooperativa', 'valor'
    }
    if not columnas_necesarias.issubset(df.columns):
        raise ValueError("Faltan columnas necesarias")

    df = df.copy()

    resultados_temp = []

    for indicador in df['ID_indicador'].unique():
        df_ind = df[df['ID_indicador'] == indicador].copy()

        # 🔹 Separar ceros (opcional pero recomendado)
        df_ind = df_ind[df_ind["valor"] > 0]

        if df_ind.empty:
            continue

        # 🔹 Transformación log
        df_ind["valor_log"] = np.log1p(df_ind["valor"])

        # 🔹 Calcular percentiles reales (umbrales)
        percentiles = [i/10 for i in range(1, 11)]
        thresholds = df_ind["valor_log"].quantile(percentiles).values

        # 🔹 Función para asignar calificación basada en thresholds
        def asignar_calificacion(valor_log):
            for i, t in enumerate(thresholds):
                if valor_log <= t:
                    return i + 1
            return 10

        df_ind["calificacion"] = df_ind["valor_log"].apply(asignar_calificacion)

        # 🔹 Promedio por cooperativa
        promedio = (
            df_ind
            .groupby('ID_cooperativa')['calificacion']
            .mean()
            .round()
            .astype(int)
        )

        for coop, calif in promedio.items():
            resultados_temp.append({
                "ID_cooperativa": str(coop),
                "ID_indicador": str(indicador),
                "calificacion": int(calif)
            })

    # 🔹 Formato final
    resultado_final = {}

    for row in resultados_temp:
        coop = row["ID_cooperativa"]
        ind = row["ID_indicador"]
        cal = row["calificacion"]

        if coop not in resultado_final:
            resultado_final[coop] = {
                "ID_cooperativa": coop,
                "calificaciones": {}
            }

        resultado_final[coop]["calificaciones"][ind] = cal

    return list(resultado_final.values())

# calcular camel

In [7]:
def calcular_camel(lista, pesos):
    # Normalizar pesos
    pesos_norm = pesos / pesos.sum()

    resultados = []

    for coop in lista:
        nombre = coop["ID_cooperativa"]
        califs = coop["calificaciones"]

        score = 0

        for indicador, peso in pesos_norm.items():
            calificacion = califs.get(indicador, 0)  # si falta, toma 0
            score += calificacion * peso

        resultados.append({
            "ID_cooperativa": nombre,
            "CAMEL_score": score
        })

    return pd.DataFrame(resultados)

# Proceso General

In [ ]:
datos_proceso=datos.copy()
def calificaciones_calculadas(df):
    calificaciones = calcular_calificaciones_formato_lista(df)
    return calificaciones